## 0. Prérequis

1. **Exécution → Modifier le type d'exécution → T4 GPU**.
2. Vérifier que le GPU est bien alloué (cellule suivante).
3. Compter **moins de 10 minutes** par entraînement.

> Toute expérience part de `src/config.py` et ne modifie **qu'un seul**
> hyperparamètre par rapport à la baseline.


In [ ]:
!nvidia-smi

## 1. Cloner le dépôt et installer les dépendances

In [ ]:
import os

REPO_URL = "https://github.com/jaouheraya12-create/transformer-hyperparam-study.git"
REPO_DIR = "transformer-hyperparam-study"

# Idempotent : on clone seulement si le dossier n'existe pas déjà.
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

%cd {REPO_DIR}
!git pull --ff-only


In [ ]:
!pip install -q -r requirements.txt

## 2. Vérifier la configuration

On vérifie que le paquet `src` est importable et que la configuration de
référence est bien celle validée par l'équipe.


In [ ]:
!python -c "import src.config; print('config OK')"

from src.config import Config

cfg = Config()
print(cfg.resume())
print("Splits :", cfg.train_size, "/", cfg.val_size, "/", cfg.test_size)


# 01 — Baseline et learning rate (Nada)

**Expériences 1 à 4.** Objectif : figer la référence puis encadrer le learning
rate de la baseline (`2e-5`) par `1e-5`, `5e-5` et `1e-4`.

| Exp. | Nom | `learning_rate` |
|---|---|---|
| 1 | `baseline` | `2e-5` |
| 2 | `lr_1e-5` | `1e-5` |
| 3 | `lr_5e-5` | `5e-5` |
| 4 | `lr_1e-4` | `1e-4` |

Tous les autres hyperparamètres restent ceux de `src/config.py`.


In [ ]:
# Vérification des données avant de lancer les runs
from src.config import Config
from src.data import build_datasets, describe_datasets
from src.utils import set_seed

cfg = Config()
set_seed(cfg.seed)
dataset, tokenizer = build_datasets(cfg)
print(describe_datasets(dataset))


## Expérience 1 — baseline

C'est la **référence** : ne rien changer. Attendre la fin du run avant de
lancer les suivants.


In [ ]:
# ~7 min sur T4
!python -m src.train --id 1 --owner Nada --name baseline


## Expériences 2 à 4 — learning rate

In [ ]:
!python -m src.train --id 2 --owner Nada --name lr_1e-5 --lr 1e-5


In [ ]:
!python -m src.train --id 3 --owner Nada --name lr_5e-5 --lr 5e-5


In [ ]:
!python -m src.train --id 4 --owner Nada --name lr_1e-4 --lr 1e-4


## Analyse : F1 en fonction du learning rate

Remplir ensuite les colonnes `train_time_s`, `train_loss`, `val_loss`,
`accuracy`, `f1` et `observations` des lignes 1 à 4 de
`experiments/experiments_plan.csv`.


In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv("results/all_experiments.csv")
mine = df[df["owner"] == "Nada"].sort_values("learning_rate")

plt.figure(figsize=(6, 4))
plt.plot(mine["learning_rate"], mine["f1"], marker="o")
plt.xscale("log")
plt.xlabel("Learning rate")
plt.ylabel("F1 (test)")
plt.title("Nada — effet du learning rate")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/nada/figures/f1_vs_lr.png", dpi=150)
plt.show()
